In [4]:
import pandas as pd
import sqlite3

# Load the two Kaggle files
df_freq = pd.read_csv('../data/freMTPL2freq.csv')
df_sev = pd.read_csv('../data/freMTPL2sev.csv')

# Merge them on the policy ID column (commonly 'IDpol')
df = df_freq.merge(df_sev[['IDpol', 'ClaimAmount']], on='IDpol', how='left')

# Fill missing ClaimAmount with 0 for policies that had no claims
df['ClaimAmount'] = df['ClaimAmount'].fillna(0)

# Sanity check
print(f"Total rows: {df.shape[0]}")
print(df[['IDpol', 'Exposure', 'ClaimNb', 'ClaimAmount']].head())

# Optional: save the merged file for future use
df.to_csv('../data/freMTPL2.csv', index=False)

# Create SQLite database and load the table
conn = sqlite3.connect('../data/motor_insurance.db')
df.to_sql('policies', conn, if_exists='replace', index=False)
conn.close()

print("✅ Database 'motor_insurance.db' created with 'policies' table.")


Total rows: 679513
   IDpol  Exposure  ClaimNb  ClaimAmount
0    1.0      0.10        1          0.0
1    3.0      0.77        1          0.0
2    5.0      0.75        1          0.0
3   10.0      0.09        1          0.0
4   11.0      0.84        1          0.0
✅ Database 'motor_insurance.db' created with 'policies' table.


In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/motor_insurance.db')

query1 = '''
SELECT
    COUNT(*) AS total_policies,
    SUM(Exposure) AS total_exposure,
    SUM(ClaimNb) AS total_claims,
    ROUND(CAST(SUM(ClaimNb) AS REAL) / SUM(Exposure), 3) AS overall_claim_frequency
FROM policies;
'''
df1 = pd.read_sql(query1, conn)
print(df1)

conn.close()

   total_policies  total_exposure  total_claims  overall_claim_frequency
0          679513   359515.229161         39788                    0.111


In [6]:
import sqlite3
import pandas as pd

# Connect to your database
conn = sqlite3.connect('../data/motor_insurance.db')

# ----- Query 1: Overall Portfolio KPIs -----
query1 = '''
SELECT
    COUNT(*) AS total_policies,
    SUM(Exposure) AS total_exposure,
    SUM(ClaimNb) AS total_claims,
    ROUND(CAST(SUM(ClaimNb) AS REAL) / SUM(Exposure), 3) AS overall_claim_frequency
FROM policies;
'''
df1 = pd.read_sql(query1, conn)
print("Query 1 - Portfolio Headline KPIs")
print(df1.to_string(index=False))

Query 1 - Portfolio Headline KPIs
 total_policies  total_exposure  total_claims  overall_claim_frequency
         679513   359515.229161         39788                    0.111


In [7]:
query2 = '''
SELECT
    CASE
        WHEN DrivAge < 25 THEN '18-24'
        WHEN DrivAge BETWEEN 25 AND 34 THEN '25-34'
        WHEN DrivAge BETWEEN 35 AND 49 THEN '35-49'
        WHEN DrivAge BETWEEN 50 AND 64 THEN '50-64'
        ELSE '65+'
    END AS driver_age_band,
    COUNT(*) AS policy_count,
    SUM(Exposure) AS total_exposure,
    SUM(ClaimNb) AS total_claims,
    ROUND(CAST(SUM(ClaimNb) AS REAL) / SUM(Exposure), 3) AS claim_frequency,
    ROUND(AVG(CASE WHEN ClaimNb > 0 THEN ClaimAmount END), 2) AS avg_claim_amount
FROM policies
GROUP BY driver_age_band
ORDER BY MIN(DrivAge);
'''
df2 = pd.read_sql(query2, conn)
print("Query 2 - Driver Age Band")
print(df2.to_string(index=False))

Query 2 - Driver Age Band
driver_age_band  policy_count  total_exposure  total_claims  claim_frequency  avg_claim_amount
          18-24         30354    12579.261085          2702            0.215           5034.88
          25-34        141213    64767.522656          6958            0.107           1575.54
          35-49        252604   131568.246141         13898            0.106           1436.64
          50-64        182016   102480.019449         11314            0.110           1365.07
            65+         73326    48120.179831          4916            0.102           1492.67


In [8]:
query3 = '''
SELECT
    CASE
        WHEN VehAge BETWEEN 0 AND 2 THEN '0-2'
        WHEN VehAge BETWEEN 3 AND 5 THEN '3-5'
        WHEN VehAge BETWEEN 6 AND 10 THEN '6-10'
        ELSE '11+'
    END AS vehicle_age_band,
    COUNT(*) AS policy_count,
    SUM(Exposure) AS total_exposure,
    SUM(ClaimNb) AS total_claims,
    ROUND(CAST(SUM(ClaimNb) AS REAL) / SUM(Exposure), 3) AS claim_frequency,
    ROUND(AVG(CASE WHEN ClaimNb > 0 THEN ClaimAmount END), 2) AS avg_claim_amount
FROM policies
GROUP BY vehicle_age_band
ORDER BY MIN(VehAge);
'''
df3 = pd.read_sql(query3, conn)
print("Query 3 - Vehicle Age Band")
print(df3.to_string(index=False))

Query 3 - Vehicle Age Band
vehicle_age_band  policy_count  total_exposure  total_claims  claim_frequency  avg_claim_amount
             0-2        188533    79424.875347         11726            0.148           1156.31
             3-5        132798    72255.649256          7385            0.102           1584.52
            6-10        172026    99320.264579         10782            0.109           1623.03
             11+        186156   108514.439978          9895            0.091           2492.85


In [9]:
query4 = '''
SELECT
    CASE
        WHEN BonusMalus = 50 THEN '50'
        WHEN BonusMalus BETWEEN 51 AND 70 THEN '51-70'
        WHEN BonusMalus BETWEEN 71 AND 100 THEN '71-100'
        WHEN BonusMalus BETWEEN 101 AND 150 THEN '101-150'
        ELSE '151+'
    END AS bonus_malus_band,
    COUNT(*) AS policy_count,
    SUM(Exposure) AS total_exposure,
    SUM(ClaimNb) AS total_claims,
    ROUND(CAST(SUM(ClaimNb) AS REAL) / SUM(Exposure), 3) AS claim_frequency,
    ROUND(AVG(CASE WHEN ClaimNb > 0 THEN ClaimAmount END), 2) AS avg_claim_amount
FROM policies
GROUP BY bonus_malus_band
ORDER BY MIN(BonusMalus);
'''
df4 = pd.read_sql(query4, conn)
print("Query 4 - Bonus-Malus Band")
print(df4.to_string(index=False))

Query 4 - Bonus-Malus Band
bonus_malus_band  policy_count  total_exposure  total_claims  claim_frequency  avg_claim_amount
              50        384697   225569.520902         19722            0.087           1259.75
           51-70        151743    74758.085992          8992            0.120           1610.98
          71-100        135157    55516.433264          9476            0.171           2614.68
         101-150          7693     3565.640806          1516            0.425           1894.81
            151+           223      105.548197            82            0.777           2456.97


In [10]:
query5 = '''
SELECT
    Region,
    COUNT(*) AS policy_count,
    SUM(Exposure) AS total_exposure,
    SUM(ClaimNb) AS total_claims,
    ROUND(CAST(SUM(ClaimNb) AS REAL) / SUM(Exposure), 3) AS claim_frequency
FROM policies
GROUP BY Region
ORDER BY claim_frequency DESC
LIMIT 5;
'''
df5 = pd.read_sql(query5, conn)
print("Query 5 - Top 5 Regions by Claim Frequency")
print(df5.to_string(index=False))

Query 5 - Top 5 Regions by Claim Frequency
              Region  policy_count  total_exposure  total_claims  claim_frequency
               Corse          4539     1778.095399           297            0.167
Languedoc-Roussillon         35939    14793.334177          2343            0.158
       Ile-de-France         69989    30331.860174          4396            0.145
   Champagne-Ardenne          3033     1210.001509           174            0.144
            Picardie          8006     3582.820221           463            0.129


In [11]:
conn.close()